---
title: Configure noise management with Estimator
description: How to configure noise management with the IBM Quantum Estimator primitive.
---

# Configure noise management with Estimator

{/*
  DO NOT EDIT THIS CELL!!!
  This cell's content is generated automatically by a script. Anything you add
  here will be removed next time the notebook is run. To add new content, create
  a new cell before or after this one.
*/}

<Accordion>
<AccordionItem title="Package versions">

The code on this page was developed using the following requirements.
We recommend using these versions or newer.

```
qiskit-ibm-runtime~=0.47.0
```
</AccordionItem>
</Accordion>

There are several ways to manage noise, typically by using various error mitigation and error suppression techniques to avoid errors before they happen.  These techniques usually cause pre-processing overhead.  Therefore, it is important to achieve a balance between perfecting your results and ensuring that your job completes in a reasonable amount of time.

Estimator supports the following noise management techniques. See [Error mitigation and suppression techniques](error-mitigation-and-suppression-techniques) for an explanation of each. See the [Custom noise management settings](#advanced-error) section for examples of enabling these techniques.

- [dynamical decoupling](/docs/api/qiskit-ibm-runtime/options-models-dynamical-decoupling-options#qiskit-ibm-runtime-options-models-dynamicaldecouplingoptions)
- [Pauli twirling](/docs/api/qiskit-ibm-runtime/options-models-twirling-options)
- [PEA](/docs/api/qiskit-ibm-runtime/options-models-zne-options)
- [PEC](/docs/api/qiskit-ibm-runtime/options-models-resilience-options#pec)
- [`resilience_level`](/docs/api/qiskit-ibm-runtime/options-models-estimator-options#resilience_level)
- [TREX](/docs/api/qiskit-ibm-runtime/options-models-resilience-options#measure_mitigation)
- [ZNE](/docs/api/qiskit-ibm-runtime/options-models-resilience-options#zne)

<span id="resilience"></span>
## Resilience level

The `resilience_level` specifies how much resilience to build against
errors. Higher levels generate more accurate results, at the expense of
longer processing times. Resilience levels can be used to configure the
cost/accuracy trade-off when applying noise management to your primitive
query. Noise management reduces errors (bias) in results by processing
the outputs from a collection, or ensemble, of related circuits. The
degree of error reduction depends on the method applied. The resilience
level abstracts the detailed choice of noise management method to allow
users to reason about the cost/accuracy trade that is appropriate to
their application.

Given this, each level corresponds to a method or methods with
increasing level of quantum sampling overhead to enable you experiment
with different time-accuracy tradeoffs. The following table shows you
which levels and corresponding methods are available for each of the
primitives.

<span id="resilience-table"></span>

| Resilience Level | Description                                                                                            | Technique                                                                 |
|------------------|-------------------------------------------------------------------------------------------------------|---------------------------------------------------------------------------|
| 0                | No mitigation                                                                                         | None                                                                      |
| 1 [Default]      | Minimal mitigation costs: Mitigate error associated with readout errors                               | [Twirled Readout Error eXtinction (TREX)](/docs/guides/error-mitigation-and-suppression-techniques#trex) measurement twirling             |
| 2                | Medium mitigation costs. Typically reduces bias in estimators, but is not guaranteed to be zero-bias. | Level 1 + [Zero Noise Extrapolation (ZNE)](/docs/guides/error-mitigation-and-suppression-techniques#zne) and gate twirling |


<Admonition type="info" title="Attention">

Resilience levels are currently in beta so sampling overhead and
solution quality will vary from circuit to circuit. New features,
advanced options, and management tools will be released on a rolling
basis. Specific noise management methods are not guaranteed to be
applied at each resilience level.
</Admonition>

### Configure Estimator with resilience levels

You can use resilience levels to specify noise management techniques, or you can set custom techniques individually as described in [Custom noise management settings](#advanced-error).

<Admonition type="note" title="Important">
Any options you manually specify in addition to the resilience level are applied in addition to the base set of options defined by the resilience level. Therefore, in principle, you could set the resilience level to 1, but then turn off measurement mitigation, although this is not advised.

For example, setting the resilience level to 0 turns off `zne_mitigation`, but `estimator.options.resilience.zne_mitigation = True` overrides that value.
</Admonition>

### Example

The following code enables ZNE, TREX, and gate twirling by
setting `resilience_level 2`.

In [1]:
from qiskit_ibm_runtime import QiskitRuntimeService

# Legacy server-side primitive before qiskit-ibm-runtime 0.50.0
# from qiskit_ibm_runtime import Estimator

# Client-side primitive introduced in qiskit-ibm-runtime 0.50.0
from qiskit_ibm_runtime.executor_estimator import Estimator

service = QiskitRuntimeService()
backend = service.least_busy(operational=True, simulator=False)

# Setting options during primitive initialization
estimator = Estimator(backend, options={"resilience_level": 2})

<span id="advanced-error"></span>
## Custom noise management settings

You can turn on and off individual noise management methods by using the [Estimator options](/docs/guides/estimator-options).

<Admonition type="note">
Not all options work together on all types of circuits.  See the [feature compatibility table](estimator-options#options-compatibility-table) for details.
</Admonition>

### Dynamical decoupling

Dynamical decoupling can be enabled by setting `enable` to `True` in the [dynamical decoupling options](/docs/api/qiskit-ibm-runtime/options-models-dynamical-decoupling-options). The `sequence_type` option can be used to pick from several different pulse sequences. The default sequence type is `"XX"`.

The following code cell shows how to enable dynamical decoupling for Estimator and choose a dynamical decoupling sequence.

In [ ]:
estimator = Estimator(mode=backend)
options = estimator.options
options.dynamical_decoupling.enable = True
options.dynamical_decoupling.sequence_type = "XpXm"

### Pauli twirling

Pauli twirling can be enabled by setting `enable_gates` to `True` in the [twirling options](/docs/api/qiskit-ibm-runtime/options-models-twirling-options). Other notable options include:

- `num_randomizations`: The number of circuit instances to draw from the ensemble of twirled circuits.
- `shots_per_randomization`: The number of shots to sample from each circuit instance.

The following code cell shows how to enable Pauli twirling and set these options for Estimator. None of these options are required to be set explicitly.

In [ ]:
estimator = Estimator(mode=backend)
options = estimator.options
options.twirling.enable_gates = True
options.twirling.num_randomizations = 32
options.twirling.shots_per_randomization = 100

### TREX

There are multiple ways to enable TREX.

#### Enable TREX by using the `measure_mitigation` option

TREX can be enabled by setting `measure_mitigation` to `True` in the [IBM Quantum Compute resilience options](/docs/api/qiskit-ibm-runtime/options-models-resilience-options) for Estimator. Options for measurement noise learning are described [here](/docs/api/qiskit-ibm-runtime/options-models-measure-noise-learning-options). As with gate twirling, you can set the number of circuit randomizations and the number of shots per randomization.

The following code cell shows how to enable TREX and set these options for Estimator. None of these options have to be set explicitly.

In [ ]:
estimator = Estimator(mode=backend)
options = estimator.options
options.resilience.measure_mitigation = True
options.resilience.measure_noise_learning.num_randomizations = 32

#### Enable TREX by using the `resilience_level` option

You can enable TREX by setting the [resilience level](/docs/guides/estimator-noise-management#resilience) to `1`, as shown in the following example.

In [ ]:
from qiskit_ibm_runtime import QiskitRuntimeService

# Legacy server-side primitive before qiskit-ibm-runtime 0.50.0
# from qiskit_ibm_runtime import Estimator

# Client-side primitive introduced in qiskit-ibm-runtime v0.50.0
from qiskit_ibm_runtime.executor_estimator import Estimator

service = QiskitRuntimeService()
backend = service.least_busy(operational=True, simulator=False)

# Setting options during primitive initialization
estimator = Estimator(backend, options={"resilience_level": 1})

### ZNE

There are multiple ways to enable ZNE.

#### Enable ZNE by using the `zne_mitigation` option

ZNE can be enabled by setting `zne_mitigation` to `True` in the [IBM Quantum Compute resilience options](/docs/api/qiskit-ibm-runtime/options-models-resilience-options) for Estimator.
The options for ZNE are described [here](/docs/api/qiskit-ibm-runtime/options-models-zne-options). The following options are notable:

- `noise_factors`: The noise factors to use for noise amplification.
- `extrapolator`: The functional form to use for the extrapolation.

The following code cell shows how to enable ZNE and set these options for Estimator. None of these options have to be set explicitly.

In [ ]:
estimator = Estimator(backend)
options = estimator.options
options.resilience.zne_mitigation = True
options.resilience.zne.noise_factors = (1, 3, 5)
options.resilience.zne.extrapolator = "exponential"

#### Enable ZNE by using the `resilience_level` option

You can enable ZNE (along with TREX and gate twirling) by setting the [resilience level](/docs/guides/estimator-noise-management#resilience) to `2`, as shown in the following example.

In [ ]:
from qiskit_ibm_runtime import QiskitRuntimeService

# Legacy server-side primitive before qiskit-ibm-runtime 0.50.0
# from qiskit_ibm_runtime import Estimator

# Client-side primitive introduced in qiskit-ibm-runtime 0.50.0
from qiskit_ibm_runtime.executor_estimator import Estimator

service = QiskitRuntimeService()
backend = service.least_busy(operational=True, simulator=False)

# Setting options during primitive initialization
estimator = Estimator(backend, options={"resilience_level": 2})

### PEA

Because PEA is a ZNE noise amplification technique, you also need to enable ZNE by setting `resilience.zne_mitigation = True`. Other [`resilience.zne`](/docs/api/qiskit-ibm-runtime/options-models-zne-options) options can additionally be used to set extrapolators, amplification levels, and so on. PEA requires a noise model, which is automatically generated when using primitives.

<Admonition type="note" title="Noise learning required">
If you are using the new client-side Estimator introduced in `qiskit-ibm-runtime` v0.50.0, a separate noise-learning job is required to use PEA.
See [Migrate from server-side to client-side Sampler and Estimator](/docs/guides/migrate-to-client-side-primitives) for more information.
</Admonition>

The following code provides an example where PEA is used to mitigate the result of an Estimator job:

In [ ]:
estimator = Estimator(backend)
options = estimator.options
options.resilience.zne_mitigation = True
options.resilience.zne.amplifier = "pea"
options.resilience.layer_noise_model = <learned noise model>

### PEC

PEC can be enabled by setting `pec_mitigation` to `True` in the [IBM Quantum Compute resilience options](/docs/api/qiskit-ibm-runtime/options-models-resilience-options) for Estimator.
The options for PEC are described [here](/docs/api/qiskit-ibm-runtime/options-models-pec-options). A limit on the sampling overhead can be set using the `max_overhead` option. Note that limiting the sampling overhead can cause the precision of the result to exceed the requested precision. The default value of `max_overhead` is 100.

<Admonition type="note" title="Noise learning required">
If you are using the new client-side Estimator introduced in `qiskit-ibm-runtime` v0.50.0, a separate noise-learning job is required to use PEC.
See [Migrate from server-side to client-side Sampler and Estimator](/docs/guides/migrate-to-client-side-primitives) for more information.
</Admonition>

The following code shows how to enable PEC and set the `max_overhead` option for Estimator.

In [ ]:
estimator = Estimator(backend)
options = estimator.options
options.resilience.pec_mitigation = True
options.resilience.pec.max_overhead = 100
options.resilience.layer_noise_model = <learned noise model>

### Full example with multiple options

In [2]:
from qiskit_ibm_runtime import QiskitRuntimeService

# Legacy server-side primitive before qiskit-ibm-runtime 0.50.0
# from qiskit_ibm_runtime import Estimator

# Client-side primitive introduced in qiskit-ibm-runtime 0.50.0
from qiskit_ibm_runtime.executor_estimator import Estimator

service = QiskitRuntimeService()
backend = service.least_busy(operational=True, simulator=False)

estimator = Estimator(backend)
options = estimator.options
# Turn on gate twirling.
options.twirling.enable_gates = True
# Turn on measurement error mitigation.
options.resilience.measure_mitigation = True

print(
    f">>> gate twirling is turned on: {estimator.options.twirling.enable_gates}"
)
print(
    f">>> measurement error mitigation is turned on: "
    f"{estimator.options.twirling.enable_gates}"
)

>>> gate twirling is turned on: True
>>> measurement error mitigation is turned on: True


## Turn off all error mitigation

For instructions to turn off all error mitigation see the [Turn off all error suppression and mitigation](estimator-options#no-error-mitigation) section in the Estimator options guide.

## Next steps

<Admonition type="tip" title="Recommendations">
  - Walk through an example that uses error mitigation in the [Cost function lesson](/learning/courses/variational-algorithm-design/cost-functions) in IBM Quantum Learning.
  - Learn more about [error mitigation and error suppression techniques](error-mitigation-and-suppression-techniques).
  - Explore [Estimator options](/docs/guides/estimator-options).
  - Decide what [execution mode](/docs/guides/execution-modes) to run your job in.
</Admonition>